In [ ]:
import numpy as np
import xarray as xr
import pandas as pd

import sys
sys.path.append('./resources/library')
from tropical_cyclone.utils import round_to_grid

/Users/davide/anaconda3/envs/fmlenv/lib/python3.11/site-packages/torch/cuda/__init__.py:58: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/Users/davide/anaconda3/envs/fmlenv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
ds = xr.open_mfdataset('path-to-sample-global-map.nc')
ibtracs = pd.read_csv('path-to/ibtracs-dataset-obtained/with-filtering', index_col=0)

/var/folders/np/mdnv6_7j1gs11_gvk6l42b5r0000gn/T/ipykernel_7710/1281608346.py:2: DtypeWarning: Columns (173,174) have mixed types. Specify dtype option on import or set low_memory=False.
  ibtracs = pd.read_csv('/Users/davide/Developer/tropical_cyclone_tracking/data/ibtracs/filtered/global_ibtracs_main-tracks_6h_1980-2022_TS-NR-ET-MX-SS-DS.csv', index_col=0)


In [3]:
# get latitudes and longitudes
lats, lons = ds['lat'].data, ds['lon'].data

In [ ]:
georef = ibtracs.copy()
georef = georef[['ISO_TIME','SID','LAT','LON','BASIN']]
georef['RLAT'] = round_to_grid(georef['LAT'], grid_res=0.25)
georef['RLON'] = round_to_grid(georef['LON'], grid_res=0.25)
georef['YLAT'] = [np.argwhere(lats==l)[0][0] for l in georef['RLAT']]
georef['XLON'] = [np.argwhere(lons==l)[0][0] for l in georef['RLON']]
georef

,ISO_TIME,SID,LAT,LON,BASIN,RLAT,RLON,YLAT,XLON
0,1980-01-01 00:00:00,1979357S10095,-16.1,81.7,SI,-16.00,81.75,344,327
1,1980-01-01 06:00:00,1979357S10095,-16.1,81.0,SI,-16.00,81.00,344,324
2,1980-01-01 12:00:00,1979357S10095,-16.1,79.9,SI,-16.00,80.00,344,320
3,1980-01-01 18:00:00,1979357S10095,-16.1,78.6,SI,-16.00,78.50,344,314
4,1980-01-02 00:00:00,1979357S10095,-16.0,77.2,SI,-16.00,77.25,344,309
...,...,...,...,...,...,...,...,...,...
143610,2022-12-24 06:00:00,2022356N09085,10.4,83.7,NI,10.50,83.75,238,335
143611,2022-12-24 12:00:00,2022356N09085,9.7,83.0,NI,9.75,83.00,241,332
143612,2022-12-24 18:00:00,2022356N09085,9.3,82.5,NI,9.25,82.50,243,330
143613,2022-12-25 00:00:00,2022356N09085,9.0,82.2,NI,9.00,82.25,244,329


In [ ]:
georef.to_csv('/global_georef.csv')